# 🗺️ 千葉県 37市 気象レーダーダッシュボード

* **作成者（グループ3人）**: [坂井 優希]、[長谷川 大祐]、[福田 直生]
* **作品概要**: 
  千葉県内の全37市を6つの地域ブロックに分類し、Open-Meteo APIから取得した最新の気象データ（天気・最高/最低気温・風速など）をインタラクティブなFoliumマップ上に可視化したダッシュボードです。プルダウンによる自動ズーム、週間予報ポップアップ、お出かけ判定スタンプ、地域ブロック別の平均気温比較機能を備えています。
* **天気データの出典URL**: 
  [Open-Meteo API](https://open-meteo.com/)
* **自分たちで追加した機能や特徴**:
  * 全37市の緯度・経度を街の中心部にフィットするよう調整し、視認性を向上。
  * 天候コードに応じたマーカーの自動色分け（晴れ・曇り・雨など）。
  * 選択した市の今日から3日間の天気・気温・風速がわかるポップアップ表示。
  * その日の気温や降水量から自動で判定される「お出かけ判定スタンプ」および「服装アドバイス」機能。
  * 千葉県内の最高気温・最低気温のトピックスおよび、6つの地域ブロックごとの平均気温（最高・最低）をひと目で比較できるパネルの搭載。
* **これから改善したいこと、追加したい機能**:
  * リアルタイムの降水レーダー（雨雲レーダー）レイヤーのより安定した常時表示。
  * 各都市の風向きを地図上のマーカーにより直感的に表現する機能の改良。

In [2]:
%pip install -q folium ipywidgets requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, HTML
import folium
from folium import plugins
import requests

# 千葉県内の全37市と緯度・経度データ
REGIONS = {
    "東葛飾ブロック": [
        {"name": "柏市", "lat": 35.8624, "lon": 139.9715},
        {"name": "松戸市", "lat": 35.7847, "lon": 139.8986},
        {"name": "野田市", "lat": 35.9511, "lon": 139.8611},
        {"name": "流山市", "lat": 35.8500, "lon": 139.9050},
        {"name": "我孫子市", "lat": 35.8689, "lon": 140.0125},
        {"name": "船橋市", "lat": 35.6944, "lon": 140.0861},
        {"name": "市川市", "lat": 35.7225, "lon": 139.9297},
        {"name": "鎌ケ谷市", "lat": 35.7778, "lon": 139.9981},
        {"name": "浦安市", "lat": 35.6483, "lon": 139.9036},
    ],
    "千葉・中央ブロック": [
        {"name": "千葉市", "lat": 35.6074, "lon": 140.1065},
        {"name": "市原市", "lat": 35.5097, "lon": 140.1200},
        {"name": "習志野市", "lat": 35.6889, "lon": 140.0247},
        {"name": "八千代市", "lat": 35.7267, "lon": 140.1133},
        {"name": "四街道市", "lat": 35.6667, "lon": 140.1667},
    ],
    "印旛ブロック": [
        {"name": "成田市", "lat": 35.7767, "lon": 140.3178},
        {"name": "佐倉市", "lat": 35.7139, "lon": 140.2247},
        {"name": "印西市", "lat": 35.8333, "lon": 140.1333},
        {"name": "白井市", "lat": 35.7853, "lon": 140.0558},
        {"name": "富里市", "lat": 35.7333, "lon": 140.3333},
        {"name": "八街市", "lat": 35.6525, "lon": 140.3175},
    ],
    "香取・東総ブロック": [
        {"name": "銚子市", "lat": 35.7347, "lon": 140.8269},
        {"name": "旭市", "lat": 35.7250, "lon": 140.6558},
        {"name": "匝瑳市", "lat": 35.6922, "lon": 140.6111},
        {"name": "香取市", "lat": 35.8942, "lon": 140.4986},
    ],
    "山武・長生・夷隅ブロック": [
        {"name": "東金市", "lat": 35.5564, "lon": 140.3900},
        {"name": "茂原市", "lat": 35.4294, "lon": 140.2936},
        {"name": "山武市", "lat": 35.6350, "lon": 140.3800},
        {"name": "大網白里市", "lat": 35.5233, "lon": 140.3194},
        {"name": "勝浦市", "lat": 35.1508, "lon": 140.3150},
        {"name": "いすみ市", "lat": 35.2530, "lon": 140.3900},
    ],
    "房総・南部ブロック": [
        {"name": "木更津市", "lat": 35.3817, "lon": 139.9275},
        {"name": "君津市", "lat": 35.3264, "lon": 139.8931},
        {"name": "富津市", "lat": 35.3094, "lon": 139.8667},
        {"name": "袖ケ浦市", "lat": 35.4358, "lon": 139.9575},
        {"name": "館山市", "lat": 34.9878, "lon": 139.8631},
        {"name": "鴨川市", "lat": 35.1056, "lon": 140.1017},
        {"name": "南房総市", "lat": 35.0350, "lon": 139.8350},
    ],
}


def interpret_weather(code):
  if code in [0, 1]:
    return "晴れ ☀️", "#ff7f0e"
  elif code in [2, 3]:
    return "曇り ☁️", "#7f7f7f"
  elif code in [51, 53, 55, 61, 63, 65, 80, 81, 82]:
    return "雨 🌧️", "#1f77b4"
  elif code in [71, 73, 75, 77, 85, 86]:
    return "雪 ❄️", "#aec7e8"
  else:
    return "その他 🌈", "#2ca02c"


def get_wind_direction(deg):
  directions = [
      "北北東",
      "北東",
      "東北東",
      "東",
      "東南東",
      "南東",
      "南南東",
      "南",
      "南南西",
      "南西",
      "西南西",
      "西",
      "西北西",
      "北西",
      "北北西",
      "北",
  ]
  try:
    deg = float(deg)
    idx = int((deg + 11.25) / 22.5) % 16
    return directions[idx]
  except Exception:
    return "不明"


# 全37市のリスト化とデータ一括取得（タイムアウト対策付き）
all_cities = []
for region_name, cities in REGIONS.items():
  for city in cities:
    all_cities.append((region_name, city))

lats = [str(c["lat"]) for _, c in all_cities]
lons = [str(c["lon"]) for _, c in all_cities]

url = (
    f"https://api.open-meteo.com/v1/forecast?"
    f"latitude={','.join(lats)}&longitude={','.join(lons)}&"
    f"daily=weather_code,temperature_2m_max,temperature_2m_min,precipitation_sum,"
    f"wind_speed_10m_max,wind_direction_10m_dominant&"
    f"timezone=Asia%2FTokyo&past_days=1&forecast_days=3"
)

try:
  response = requests.get(url, timeout=15)
  data = response.json()
except Exception as e:
  print(f"⚠️ API通信エラーが発生しました: {e}")
  data = {}

data_list = data if isinstance(data, list) else [data]

city_weather_map = {}
flat_city_list = []
today_str = datetime.now().strftime("%Y-%m-%d")

for i, (region_name, city) in enumerate(all_cities):
  target_data = data_list[i] if i < len(data_list) else {}
  daily = target_data.get("daily", {})

  times = daily.get("time", [])
  codes = daily.get("weather_code", [])
  t_max = daily.get("temperature_2m_max", [])
  t_min = daily.get("temperature_2m_min", [])
  precip = daily.get("precipitation_sum", [])
  wind_speed = daily.get("wind_speed_10m_max", [])
  wind_dir = daily.get("wind_direction_10m_dominant", [])

  if not isinstance(times, list):
    times = [times]
  if not isinstance(codes, list):
    codes = [codes]
  if not isinstance(t_max, list):
    t_max = [t_max]
  if not isinstance(t_min, list):
    t_min = [t_min]
  if not isinstance(precip, list):
    precip = [precip]
  if not isinstance(wind_speed, list):
    wind_speed = [wind_speed]
  if not isinstance(wind_dir, list):
    wind_dir = [wind_dir]

  daily_forecasts = []
  today_wind_text = "無風"
  today_max_temp = 0.0
  today_min_temp = 0.0
  today_precip = 0.0
  today_code = 0

  for d_idx, date_str in enumerate(times):
    w_code = codes[d_idx] if d_idx < len(codes) and codes[d_idx] is not None else 0
    w_desc, w_color = interpret_weather(w_code)

    dt_obj = datetime.strptime(str(date_str), "%Y-%m-%d")
    formatted_date = f"{dt_obj.month}月{dt_obj.day}日"

    ws = (
        wind_speed[d_idx]
        if d_idx < len(wind_speed) and wind_speed[d_idx] is not None
        else 0.0
    )
    wd = (
        wind_dir[d_idx]
        if d_idx < len(wind_dir) and wind_dir[d_idx] is not None
        else 0.0
    )
    wd_str = get_wind_direction(wd)

    max_t = (
        t_max[d_idx]
        if d_idx < len(t_max) and t_max[d_idx] is not None
        else 0.0
    )
    min_t = (
        t_min[d_idx]
        if d_idx < len(t_min) and t_min[d_idx] is not None
        else 0.0
    )
    p_val = (
        precip[d_idx]
        if d_idx < len(precip) and precip[d_idx] is not None
        else 0.0
    )

    if str(date_str) == today_str:
      today_max_temp = max_t
      today_min_temp = min_t
      today_precip = p_val
      today_code = w_code
      today_wind_text = f"{wd_str} {ws}m/s"

    daily_forecasts.append({
        "date_str": formatted_date,
        "weather": w_desc,
        "max_temp": max_t,
        "min_temp": min_t,
        "wind_text": f"{wd_str} {ws}m/s",
    })

  today_color = "#ff3333"
  for f_idx, d_s in enumerate(times):
    if str(d_s) == today_str:
      w_code = (
          codes[f_idx]
          if f_idx < len(codes) and codes[f_idx] is not None
          else 0
      )
      _, c_val = interpret_weather(w_code)
      today_color = c_val
      break

  advice = "過ごしやすい陽気です♪"
  if today_max_temp >= 25:
    advice = "半袖で快適に過ごせます 👕"
  elif today_max_temp <= 12:
    advice = "しっかりとしたコートや防寒着が必要です 🧥"
  else:
    advice = "カーディガンやジャケットがあると安心 👔"

  stamp = "✨ お出かけ日和"
  if today_code in [51, 53, 55, 61, 63, 65, 80, 81, 82] or today_precip > 1.0:
    stamp = "☔ 傘必須（雨模様）"
  elif today_max_temp >= 20 and today_precip == 0:
    stamp = "🧺 洗濯・お出かけ最適"
  elif today_code in [2, 3]:
    stamp = "☁️ 曇り空（過ごしやすい）"

  city_weather_map[city["name"]] = {
      "name": city["name"],
      "region": region_name,
      "lat": city["lat"],
      "lon": city["lon"],
      "color": today_color,
      "max_temp": today_max_temp,
      "min_temp": today_min_temp,
      "wind": today_wind_text,
      "advice": advice,
      "stamp": stamp,
      "forecasts": daily_forecasts,
  }
  flat_city_list.append(city["name"])

# --- 気温ランキング & ブロック別平均気温の計算 ---
if city_weather_map:
  sorted_by_max = sorted(
      city_weather_map.values(), key=lambda x: x["max_temp"], reverse=True
  )
  hottest_city = sorted_by_max[0]

  sorted_by_min = sorted(
      city_weather_map.values(), key=lambda x: x["min_temp"], reverse=False
  )
  coldest_city = sorted_by_min[0]
else:
  hottest_city = {"name": "データなし", "max_temp": 0.0}
  coldest_city = {"name": "データなし", "min_temp": 0.0}

region_avg_temps = {}
for r_name in REGIONS.keys():
  r_cities = [
      info for info in city_weather_map.values() if info["region"] == r_name
  ]
  if r_cities:
    avg_max = round(
        sum([c["max_temp"] for c in r_cities]) / len(r_cities), 1
    )
    avg_min = round(
        sum([c["min_temp"] for c in r_cities]) / len(r_cities), 1
    )
    region_avg_temps[r_name] = {"max": avg_max, "min": avg_min}
  else:
    region_avg_temps[r_name] = {"max": 0.0, "min": 0.0}

# --- UIとマップ描画の構築 ---
output_area = widgets.Output()


def generate_map(center_lat=35.55, center_lon=140.15, zoom=9, selected_city=None):
  m = folium.Map(
      location=[center_lat, center_lon],
      zoom_start=zoom,
      tiles="https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png",
      attr="&copy; OpenStreetMap contributors",
  )
  plugins.Fullscreen(position="topright").add_to(m)

  for c_name, info in city_weather_map.items():
    forecast_html = ""
    for f in info["forecasts"]:
      forecast_html += f"""
            <tr style="border-bottom: 1px solid #ddd;">
                <td style="padding: 2px; font-size: 10px;">{f['date_str']}</td>
                <td style="padding: 2px; font-size: 10px;">{f['weather']}</td>
                <td style="padding: 2px; font-size: 10px; color: #d9534f;">{f['max_temp']}℃</td>
                <td style="padding: 2px; font-size: 10px; color: #428bca;">{f['min_temp']}℃</td>
                <td style="padding: 2px; font-size: 10px;">💨{f['wind_text']}</td>
            </tr>
            """

    popup_html = f"""
        <div style="font-family: sans-serif; width: 220px;">
            <h4 style="margin: 0 0 4px 0; color: #000; border-bottom: 2px solid #000; padding-bottom: 2px;">{info['name']}</h4>
            <p style="margin: 2px 0; font-size: 11px; color: #666;">ブロック: {info['region']}</p>
            <div style="margin: 3px 0; font-size: 11px; background: #fff3cd; padding: 4px; border-radius: 4px; border: 1px solid #ffeeba; text-align: center;"><b>{info['stamp']}</b></div>
            <p style="margin: 3px 0; font-size: 11px; background: #eef; padding: 3px; border-radius: 4px;"><b>💡服装アドバイス:</b><br>{info['advice']}</p>
            <table style="width: 100%; border-collapse: collapse; margin-top: 4px;">
                <tr style="background: #f2f2f2; font-size: 10px;">
                    <th style="padding: 2px;">日付</th>
                    <th style="padding: 2px;">天気</th>
                    <th style="padding: 2px;">最高</th>
                    <th style="padding: 2px;">最低</th>
                    <th style="padding: 2px;">風</th>
                </tr>
                {forecast_html}
            </table>
        </div>
        """

    r = 8 if c_name == selected_city else 5.5

    folium.CircleMarker(
        location=[info["lat"], info["lon"]],
        radius=r,
        color="#000000",
        weight=1.5,
        fill=True,
        fill_color=info["color"],
        fill_opacity=0.9,
        popup=folium.Popup(popup_html, max_width=240),
        tooltip=f"{c_name} (風速: {info['wind']})",
    ).add_to(m)

  return m


def update_display(city_name):
  with output_area:
    output_area.clear_output()
    if city_name in city_weather_map:
      info = city_weather_map[city_name]
      print(f"=== 【{info['name']}】（{info['region']}）の気象データ ===")

      m = generate_map(
          center_lat=info["lat"],
          center_lon=info["lon"],
          zoom=11,
          selected_city=city_name,
      )
      display(m)
    else:
      print("⚠️ 指定された市が見つかりません。")


# プルダウンメニューウィジェット
dropdown_options = [(name, name) for name in flat_city_list]
dropdown = widgets.Dropdown(
    options=dropdown_options,
    value="柏市",
    description="市を選択:",
    style={"description_width": "initial"},
)


def on_dropdown_change(change):
  if change["type"] == "change" and change["name"] == "value":
    update_display(change["new"])


dropdown.observe(on_dropdown_change)

# 初期表示
update_display("柏市")

# ブロック別平均気温パネルのHTML生成
block_cards_html = ""
for r_name, t_dict in region_avg_temps.items():
  block_cards_html += f"""
    <div style="flex: 1; background: #fff; padding: 6px; margin: 2px; border-radius: 4px; border: 1px solid #ddd; text-align: center; min-width: 110px;">
        <div style="font-size: 10px; color: #666; font-weight: bold;">{r_name}</div>
        <div style="font-size: 12px; color: #d9534f; margin-top: 2px;">最高: <b>{t_dict['max']}℃</b></div>
        <div style="font-size: 12px; color: #428bca;">最低: <b>{t_dict['min']}℃</b></div>
    </div>
    """

panel_html = f"""
<div style="background: #f9f9f9; padding: 10px; border-radius: 6px; border: 1px solid #ddd; margin-bottom: 10px; font-family: sans-serif;">
    <div style="font-size: 13px; margin-bottom: 6px;">
        <b>🔥 本日の千葉県 気温トピックス:</b>
        <span style="color: #d9534f; margin-left: 10px;">最高気温: <b>{hottest_city['name']}</b> ({hottest_city['max_temp']}℃)</span>
        <span style="color: #428bca; margin-left: 15px;">最低気温: <b>{coldest_city['name']}</b> ({coldest_city['min_temp']}℃)</span>
    </div>
    <div style="font-size: 12px; font-weight: bold; margin-bottom: 4px; color: #333;">📊 地域ブロック別 平均気温比較（最高・最低）:</div>
    <div style="display: flex; flex-wrap: wrap; gap: 2px;">
        {block_cards_html}
    </div>
</div>
"""

# 画面に配置
display(
    widgets.VBox(
        [
            widgets.HTML("<h3>🗺️ 千葉県 37市 気象レーダー</h3>"),
            widgets.HTML(panel_html),
            dropdown,
            widgets.HTML("<hr>"),
            output_area,
        ]
    )
)